### 1) 환경변수 설정

In [1]:
from dotenv import load_dotenv
load_dotenv()


True

### 2) documents.jsonl 로드 → LangChain Document 변환

In [ ]:
# import json
# from pathlib import Path
# from langchain_core.documents import Document

# JSONL_PATH = "./out/documents.jsonl"

# docs = []
# with open(JSONL_PATH, "r", encoding="utf-8") as f:
#     for line in f:
#         obj = json.loads(line)
#         docs.append(Document(page_content=obj["content"], metadata=obj["metadata"]))

# len(docs), docs[0].metadata


(592,
 {'document_id': '20241001798_0.0_한영대학한영대학교특성화맞춤형교육환경구축트랙운영학사정보',
  'row_index': 0,
  '공고 번호': '20241001798',
  '공고 차수': 0.0,
  '사업명': '한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보시스템 고도화',
  '발주 기관': '한영대학',
  '공개 일자': '2024-10-04 13:51:23',
  '입찰 참여 시작일': 'nan',
  '입찰 참여 마감일': '2024-10-15 17:00:00',
  '사업 금액': 130000000.0,
  '파일형식': 'hwp',
  '파일명': '한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
  'raw_path': 'data\\raw\\한영대학_한영대학교 특성화 맞춤형 교육환경 구축 - 트랙운영 학사정보.hwp',
  'match_type': 'exact_norm',
  'type': 'summary',
  'chunk': -1,
  'source_type': 'csv_summary'})

### 3) 임베딩 모델 준비

In [2]:
from langchain_openai import OpenAIEmbeddings

emb = OpenAIEmbeddings(model="text-embedding-3-small")

### 4) FAISS 인덱스 생성 + 저장

In [ ]:
# from langchain_community.vectorstores import FAISS

# FAISS_DIR = "./out/faiss_index"

# # (옵션) 문서 수가 많으면 배치로 나눠서 안정적으로
# batch_size = 500

# db = None
# for i in range(0, len(docs), batch_size):
#     batch = docs[i:i+batch_size]
#     if db is None:
#         db = FAISS.from_documents(batch, emb)
#     else:
#         db.add_documents(batch)

# db.save_local(FAISS_DIR)
# print("Saved FAISS to:", FAISS_DIR)


Saved FAISS to: ./out/faiss_index


### 5) 저장된 FAISS 로드 + Retriver 구성

In [4]:
from langchain_community.vectorstores import FAISS

FAISS_DIR = "./out/faiss_merged_hwp_pdf_cs600_ov100"

db = FAISS.load_local(FAISS_DIR, emb, allow_dangerous_deserialization=True)

# 중복 줄이려면 MMR이 보통 더 좋습니다.
retriever = db.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 12, 
        "fetch_k": 80, 
        "lambda_mult": 0.3
        },
)

### 6) rerank

In [5]:
from flashrank import Ranker, RerankRequest

base_retriever = retriever  # 이미 만든 MMR retriever
ranker = Ranker(model_name="ms-marco-MiniLM-L-12-v2")
TOP_N = 8

def rerank_docs(question: str):
    docs = base_retriever.invoke(question)  # 후보 12개
    if not docs:
        return []

    passages = [{"id": i, "text": d.page_content} for i, d in enumerate(docs)]
    ranked = ranker.rerank(RerankRequest(query=question, passages=passages))

    top_ids = [r["id"] for r in ranked[:TOP_N]]
    return [docs[i] for i in top_ids]

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)


### 6) RAG 프롬프트 + 생성 모델 + 체인 구성

In [ ]:
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.runnables import RunnableLambda, RunnablePassthrough
# from langchain_core.output_parsers import StrOutputParser

# llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

# prompt = ChatPromptTemplate.from_messages([
#     ("system",
#      "당신은 공공입찰/RFP 문서를 근거로 답변하는 어시스턴트입니다.\n"
#      "반드시 제공된 CONTEXT(발췌문) 안에서만 답하세요. CONTEXT 밖의 지식/추론/추측은 금지합니다.\n"
#      "CONTEXT에 근거가 없으면 정확히 다음 문장만 출력하세요: 근거 문서에서 확인할 수 없습니다\n\n"
#      "출력 형식(반드시 준수):\n"
#      "1) ANSWER: 한글로 핵심만\n"
#      "2) SOURCES: 아래 형식으로 bullet(최소 1개)\n"
#      "   - file_name=<...> | doc_id=<...> | chunk=<...> | source_type=<...>\n"
#      "SOURCES에는 CONTEXT에 포함된 메타만 사용하세요. 임의로 만들지 마세요."),
#     ("human",
#      "질문: {question}\n\n"
#      "CONTEXT:\n{context}")
# ])

# def format_docs(docs, max_chars=1600):
#     blocks = []
#     for d in docs:
#         md = d.metadata or {}
#         blocks.append(
#             "-----\n"
#             f"[file_name={md.get('file_name')}] "
#             f"[doc_id={md.get('doc_id') or md.get('document_id')}] "
#             f"[chunk={md.get('chunk')}] "
#             f"[source_type={md.get('source_type')}]\n"
#             f"{d.page_content[:max_chars]}"
#         )
#     return "\n\n".join(blocks)

# rag_chain = (
#     {
#         "question": RunnablePassthrough(),
#         "context": RunnableLambda(rerank_docs) | RunnableLambda(format_docs),
#     }
#     | prompt
#     | llm
#     | StrOutputParser()
# )


In [6]:
import json
from typing import Any, Dict, List, Optional

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import (
    RunnableLambda,
    RunnablePassthrough,
    RunnableBranch,
)
from langchain_core.output_parsers import StrOutputParser


# =========================
# 0) LLM / Prompt
# =========================
llm = ChatOpenAI(model="gpt-5-mini", temperature=0)

prompt = ChatPromptTemplate.from_messages([
    ("system",
     "당신은 공공입찰/RFP 문서를 근거로 답변하는 어시스턴트입니다.\n"
     "반드시 제공된 CONTEXT(발췌문) 안에서만 답하세요. CONTEXT 밖의 지식/추론/추측은 금지합니다.\n"
     "CONTEXT에 근거가 없으면 정확히 다음 문장만 출력하세요: 근거 문서에서 확인할 수 없습니다\n"
     "(이 경우 ANSWER/SOURCES 출력 금지)\n\n"
     "출력 형식(반드시 준수):\n"
     "1) ANSWER: 한글로 핵심만\n"
     "2) SOURCES: 아래 형식으로 bullet(최소 1개)\n"
     "   - file_name=<...> | doc_id=<...> | chunk=<...> | source_type=<...>\n"
     "SOURCES에는 CONTEXT에 포함된 META만 사용하세요. 임의로 만들지 마세요.\n"
     "가능하면 숫자/기간/요건은 원문 표현을 최대한 유지하세요."),
    ("human",
     "질문: {question}\n\n"
     "CONTEXT:\n{context}")
])

NO_EVIDENCE = "근거 문서에서 확인할 수 없습니다"


# =========================
# 1) 메타 안정화 유틸
# =========================
def _first_non_empty(*vals):
    for v in vals:
        if v is None:
            continue
        if isinstance(v, str) and not v.strip():
            continue
        return v
    return None

def normalize_meta(md: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    md = md or {}

    file_name = _first_non_empty(
        md.get("file_name"),
        md.get("filename"),
        md.get("source"),          # 어떤 파이프라인은 source에 파일 경로가 들어감
        md.get("path"),
    )

    doc_id = _first_non_empty(
        md.get("doc_id"),
        md.get("document_id"),
        md.get("id"),
    )

    chunk = _first_non_empty(
        md.get("chunk"),
        md.get("chunk_id"),
        md.get("chunk_index"),
        md.get("chunk_no"),
    )

    source_type = _first_non_empty(
        md.get("source_type"),
        md.get("type"),
        md.get("parser"),
    )

    page = _first_non_empty(
        md.get("page"),
        md.get("page_number"),
        md.get("page_index"),
    )

    section = _first_non_empty(
        md.get("section"),
        md.get("section_path"),
        md.get("heading"),
    )

    norm = {
        "file_name": file_name,
        "doc_id": doc_id,
        "chunk": chunk,
        "source_type": source_type,
    }

    # 있으면 도움이 되는 옵션 메타(프롬프트는 SOURCES에 쓰지 말라고 했으니 CONTEXT 안에서만 참고용)
    if page is not None:
        norm["page"] = page
    if section is not None:
        norm["section"] = section

    return norm


# =========================
# 2) CONTEXT 포맷: META를 JSON 한 줄로 강제
# =========================
def format_docs_json_meta(docs: List[Any], max_chars_per_doc: int = 2600) -> str:
    blocks = []
    for d in docs:
        meta = normalize_meta(getattr(d, "metadata", None))
        meta_line = "META: " + json.dumps(meta, ensure_ascii=False)
        text = getattr(d, "page_content", "") or ""
        text = text[:max_chars_per_doc]
        blocks.append(f"-----\n{meta_line}\nTEXT:\n{text}")
    return "\n\n".join(blocks)


# =========================
# 3) (중요) LLM 호출 전 "docs 비었는지" 분기
# =========================
# 아래 rerank_docs는 질문 문자열을 받아 "Document 리스트"를 반환해야 합니다.
# - 이미 구현해 둔 rerank_docs(question: str) 가 있다면 그대로 사용하세요.
# - 없다면 base_retriever.invoke(question)만 반환하는 임시 구현으로도 동작합니다.

def build_inputs(question: str) -> Dict[str, Any]:
    docs = rerank_docs(question)  # <- 사용자께서 이미 만든 rerank_docs 사용
    return {
        "question": question,
        "docs": docs,
    }

def has_docs(x: Dict[str, Any]) -> bool:
    docs = x.get("docs") or []
    return len(docs) > 0

def to_llm_payload(x: Dict[str, Any]) -> Dict[str, str]:
    docs = x["docs"]
    context = format_docs_json_meta(docs)
    return {
        "question": x["question"],
        "context": context,
    }


# LLM 서브체인 (문서가 있을 때만 실행)
llm_chain = to_llm_payload | prompt | llm | StrOutputParser()

# 최종 안정화 체인: docs가 없으면 즉시 NO_EVIDENCE 반환
rag_chain = (
    RunnableLambda(build_inputs)
    | RunnableBranch(
        (RunnableLambda(has_docs), llm_chain),
        RunnableLambda(lambda _: NO_EVIDENCE),
    )
)


### 7) 실행

In [7]:
question = "부산 국제 영화제 사업 담당자의 이름은?"
answer = rag_chain.invoke(question)
print(answer)

INFO:httpx:HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


근거 문서에서 확인할 수 없습니다
